# Pose Estimator Data Organization

The first step in my project pipeline is fine-tuning the YOLO11 object-detection model on running data.

For this, I found the Athlete Pose Dataset, which is a collection of videos recording athletic actions including running, figure skating, and track and field (primarily shotput). I needed a high-quality dataset like Athlete Pose because it had both videos and matching ground truth pose coordinates.
(https://github.com/calvinyeungck/AthletePose3D)

The dataset had pose coordinates in both 2D and 3D. For my project, only 2D data is needed. Still this resulted in a full dataset of 21 GB. I wanted only the running data for fine-tuning the YOLO model, which is where this file comes in.

## Extracting COCO Annotations file

This script was used to find and extract annotations in the COCO format.

In [ ]:
import zipfile
import os

# run this in same dir of pose_2d.zip
zip_file_path = 'pose_2d.zip'
target_folder = 'pose_2d/annotations/'

with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    all_files = zip_ref.namelist()
    
    json_files_to_extract = [
        f for f in all_files 
        if target_folder in f and f.endswith('.json')
    ]
    
    if not json_files_to_extract:
        print("No JSON files found.")
    else:
        print(f"Found {len(json_files_to_extract)} JSON files. Extracting...")
        
        for file in json_files_to_extract:
            print(f"\tExtracting: {file}")
            zip_ref.extract(file)

## Interpreting the Dataset

Again, this was a 21 GB dataset when compressed, so I didn't want to expand all of the data on my machine. Instead, I wanted to use the annotations files to understand how the data is organized and selectively extract the running data.

In [2]:
import json

file_path = 'datasets/athlete_pose_annotations/train_set.json'

with open(file_path, 'r') as f:
    coco_data = json.load(f)

print("\n--- Keys in the main COCO dictionary ---")
print(coco_data.keys())


--- Keys in the main COCO dictionary ---
dict_keys(['images', 'annotations', 'categories', 'info'])


In [3]:
print("\n--- First Image Record ---")
print(json.dumps(coco_data['images'][0], indent=4))

print("\n--- First Annotations Record ---")
print(json.dumps(coco_data['annotations'][0], indent=4))

print("\n--- First Categories Record ---")
print(json.dumps(coco_data['categories'][0], indent=4))

print("\n--- First Info Record ---")
print(json.dumps(coco_data['info'][0], indent=4))


--- First Image Record ---
{
    "id": 20000000001,
    "width": 1280,
    "height": 768,
    "file_name": "20000000001.jpg"
}

--- First Annotations Record ---
{
    "keypoints": [
        578.59,
        161.58,
        2.0,
        -1.0,
        -1.0,
        0,
        -1.0,
        -1.0,
        0,
        -1.0,
        -1.0,
        0,
        -1.0,
        -1.0,
        0,
        543.76,
        221.68,
        2.0,
        624.19,
        217.52,
        2.0,
        524.82,
        294.53,
        2.0,
        688.87,
        258.61,
        2.0,
        534.79,
        299.82,
        2.0,
        659.29,
        314.97,
        2.0,
        596.05,
        340.4,
        2.0,
        625.44,
        335.92,
        2.0,
        638.17,
        499.92,
        2.0,
        575.59,
        467.55,
        2.0,
        698.72,
        600.04,
        2.0,
        573.39,
        572.64,
        2.0
    ],
    "num_keypoints": 17,
    "area": 132205.75,
    "iscrowd": 0,
    "

Looking at the annotations JSON files, it was not clear which images were running. The images were labeled with numbers, so I decided to look at a random selection of images from the training set to determine if there was a number range that all the running images were in. This was done for each of the train_set, val_set, and test_set.

In [ ]:
import random

zip_file_path = 'pose_2d.zip' 
image_folder_in_zip = 'pose_2d/test_set/'
output_dir = '../Documents/AppliedML/FinalProject/datasets/athlete_pose/pose_2d/train_set/manual_curation/'

os.makedirs(output_dir, exist_ok=True)

train_set_filenames = []

print("Reading zip index...")
with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    all_files = zip_ref.namelist()

    image_files = [f for f in all_files if f.startswith(image_folder_in_zip) and f.endswith('.jpg')]
    
    print(f"Found {len(image_files)} total images.")
    
    sample_size = min(1000, len(image_files)) 
    sampled_files = random.sample(image_files, sample_size)
    
    print(f"Extracting {sample_size} random images to {output_dir}...")
    for i, file in enumerate(sampled_files):
        filename = os.path.basename(file)
        train_set_filenames.append(filename)
        
        source = zip_ref.open(file)
        target = open(os.path.join(output_dir, filename), "wb")
        with source, target:
            target.write(source.read())
            
        if i % 100 == 0 and i > 0:
            print(f"  ...extracted {i} images")

After manually looking through the random selection of images, I found that running images were in:

train_set: 200*.jpeg and 300*.jpeg

test_set: 200*.jpeg

validation_set: 200*.jpeg

## Extracting Running Data

The next step was to extract running data from each of the train_set, val_set, and test_set. This was done by looking into each folder and pulling out all the images with the target prefixes. They were stored in separate folders based on the train/val/test split.

In [ ]:
def extract_running_images(image_folder_in_zip, output_dir, target_prefixes):
    os.makedirs(output_dir, exist_ok=True)

    print(f"Reading zip index from {zip_file_path}...")
    with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
        all_files = zip_ref.namelist()
        
        running_files = [
            f for f in all_files 
            if f.startswith(image_folder_in_zip) 
            and f.endswith('.jpg') 
            and os.path.basename(f).startswith(target_prefixes)
        ]
        
        print(f"Found {len(running_files)} running images! Extracting now...")
        
        for i, file in enumerate(running_files):
            filename = os.path.basename(file)
            
            source = zip_ref.open(file)
            target = open(os.path.join(output_dir, filename), "wb")
            with source, target:
                target.write(source.read())
                
            if (i + 1) % 500 == 0:
                print(f"  ...extracted {i + 1} images")

    print(f"{len(running_files)} running images saved to {output_dir}")

In [ ]:
zip_file_path = 'pose_2d.zip'

image_folder_in_zip = 'pose_2d/train_set/'
output_dir = '../Documents/AppliedML/FinalProject/datasets/athlete_pose/pose_2d/train_set/'
target_prefixes = ['2000', '3000']

extract_running_images(image_folder_in_zip, output_dir, target_prefixes)

In [ ]:
image_folder_in_zip = 'pose_2d/valid_set/'
output_dir = '../Documents/AppliedML/FinalProject/datasets/athlete_pose/pose_2d/valid_set/'
target_prefixes = ('2000')

extract_running_images(image_folder_in_zip, output_dir, target_prefixes)

## Reduce COCO Annotations

Now with all the running data, the original COCO annotations would have a bunch of annotations from figure skating and track videos that are no longer needed. This function takes an original annotation file and copies only the annotations with target_prefixes into a reduced annotation file.

In [ ]:
def reduce_annotations(input_path, output_path, target_prefixes):
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    print(f"Loading original annotations from {input_path}...")
    with open(input_path, 'r') as f:
        coco_data = json.load(f)

    filtered_images = []
    kept_image_ids = set()

    for img in coco_data.get('images', []):
        filename = img.get('file_name', '')
        if filename.startswith(target_prefixes):
            filtered_images.append(img)
            kept_image_ids.add(img['id'])

    filtered_annotations = [
        ann for ann in coco_data.get('annotations', [])
        if ann.get('image_id') in kept_image_ids
    ]

    new_coco_data = {
        "images": filtered_images,
        "annotations": filtered_annotations,
        "info": coco_data.get("info", {}),
        "licenses": coco_data.get("licenses", []),
        "categories": coco_data.get("categories", [])
    }

    print(f"Saving filtered data to {output_path}...")
    with open(output_path, 'w') as f:
        json.dump(new_coco_data, f)

    print("\n--- Filter Summary ---")
    print(f"Images kept: {len(filtered_images)} of {len(coco_data.get('images', []))}")
    print(f"Annotations kept: {len(filtered_annotations)} of {len(coco_data.get('annotations', []))}")

In [ ]:
input_json_path = 'datasets/athlete_pose/pose_2d/annotations/train_set.json'
output_json_path = 'datasets/athlete_pose/pose_2d/annotations/train_set_running_only.json'

target_prefixes = ('2000', '3000')

reduce_annotations(input_json_path, output_json_path, target_prefixes)

## Converting COCO to YOLO Format

The Athlete Pose dataset uses COCO format, which has all of the ground truth pose annotation in a single JSON file (the file created in the previous step). However, to fine-tune the YOLO model, the data needs to be re-organized. YOLO wants one .txt file per frame in the running dataset. The images and .txt labels are put in separate folder: images/ and labels/ within each split train/, val/, and test/ .

Fortunately, this is a common conversion and Ultralytics, the platform behind the YOLO models, have implemented a conversion function used below.

In [5]:
from ultralytics.data.converter import convert_coco

convert_coco(labels_dir='datasets/athlete_pose/pose_2d/annotations/', use_keypoints=True)

Annotations /Users/harrisonyork/Documents/AppliedML/FinalProject/datasets/athlete_pose/pose_2d/annotations/test_set.json: 100% ━━━━━━━━━━━━ 28402/28402 8.9Kit/s 3.2s0.0s
Annotations /Users/harrisonyork/Documents/AppliedML/FinalProject/datasets/athlete_pose/pose_2d/annotations/test_set_running_only.json: 100% ━━━━━━━━━━━━ 2936/2936 10.6Kit/s 0.3s.2s
Annotations /Users/harrisonyork/Documents/AppliedML/FinalProject/datasets/athlete_pose/pose_2d/annotations/train_set.json: 100% ━━━━━━━━━━━━ 71375/71375 8.6Kit/s 8.3s0.1ss
Annotations /Users/harrisonyork/Documents/AppliedML/FinalProject/datasets/athlete_pose/pose_2d/annotations/train_set_running_only.json: 100% ━━━━━━━━━━━━ 10504/10504 9.8Kit/s 1.1s.1s
Annotations /Users/harrisonyork/Documents/AppliedML/FinalProject/datasets/athlete_pose/pose_2d/annotations/valid_set.json: 100% ━━━━━━━━━━━━ 29789/29789 9.6Kit/s 3.1s<0.0s
Annotations /Users/harrisonyork/Documents/AppliedML/FinalProject/datasets/athlete_pose/pose_2d/annotations/valid_set_runni

Now the data is ready for fine-tuning the YOLO model. The converted data has the following format:

datasets/athlete_pose_converted/<br>
├── train/<br>
│&emsp;&emsp;&emsp;├── images/<br>
│&emsp;&emsp;&emsp;└── labels/<br>
├── test/<br>
│&emsp;&emsp;&emsp;├── images/<br>
│&emsp;&emsp;&emsp;└── labels/<br>
└── valid/<br>
&emsp;&emsp;&emsp;├── images/<br>
&emsp;&emsp;&emsp;└── labels/<br>